In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import scanpy as sc
import torch
from driver_genes.losses.dis import edist_euclidean

In [2]:
ad = sc.read('../datasets/schmidt/preprocessed.h5ad')
sc.pp.pca(ad, n_comps=50)
ad


AnnData object with n_obs × n_vars = 57835 × 17572
    obs: 'condition', 'cluster_name', 'CD4.or.CD8', 'perturbation', 'crispr', 'donor', 'Phase', 'num_features', 'feature_call', 'num_umis', 'n_genes'
    var: 'gene_ids', 'feature_types', 'n_cells', 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'highly_variable_raw', 'highly_variable_batch'
    uns: 'hvg', 'log1p', 'pca'
    obsm: 'X_pca'
    varm: 'PCs'
    layers: 'raw'

In [3]:
def sample_smp(X, n_samples=None):
    if n_samples is None:
        n_samples = X.shape[0]
    if n_samples > X.shape[0]:
        return X[torch.randint(0, X.shape[0], (n_samples,))]
    else:
        return X[torch.randperm(X.shape[0])[:n_samples]]

In [4]:
ct_dict = {ct: i for i, ct in enumerate(ad.obs['condition'].unique())}
pert_dict = {pert: i for i, pert in enumerate(ad.obs['perturbation'].unique()) if pert != 'control'}
pert_dict['control'] = 999
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [5]:
X = torch.from_numpy(ad[:,ad.var['highly_variable']].obsm['X_pca'].toarray()).to(device)
cti = torch.tensor([ct_dict[ct] for ct in ad.obs['condition'].values], device=device)
perti = torch.tensor([pert_dict[pert] for pert in ad.obs['perturbation'].values], device=device)

In [ ]:
distance_list = []
for rep in range(10):
    for ct in ct_dict:
        Xctrl = sample_smp(X[perti == 999], 200) # randomly sample 200 control cells
        for pert in pert_dict:
            Xpert = sample_smp(X[perti == pert_dict[pert]], 50) # randomly sample 50 cells for each perturbation
            dist = edist_euclidean(Xctrl, Xpert).cpu().item()
            distance_list.append({'cell_type': ct, 'perturbation': pert, 'distance': dist, 'rep': rep})
distance_df = pd.DataFrame(distance_list)
print(distance_df.shape)
distance_df.head()


(1400, 4)


,cell_type,perturbation,distance,rep
0,Resting,PLCG2,0.411449,0
1,Resting,OTUD7B,0.626789,0
2,Resting,CD247,1.586958,0
3,Resting,PRKD2,0.377089,0
4,Resting,IL2RB,0.613173,0


In [ ]:
distance_df = distance_df.groupby(
    ['cell_type', 'perturbation']
).agg(['mean', 'std']).reset_index().drop(columns=['rep'])
distance_df

cell_type perturbation  distance          
                                     mean       std
0    Re-stimulated       ABCB10  0.455849  0.053009
1    Re-stimulated       AKAP12  0.397815  0.081431
2    Re-stimulated         ALX4  0.590052  0.140982
3    Re-stimulated     APOBEC3C  0.770958  0.180398
4    Re-stimulated     APOBEC3D  0.902861  0.107104
..             ...          ...       ...       ...
135        Resting     TRAF3IP2  1.161419  0.494322
136        Resting       TRIM21  0.388526  0.044252
137        Resting         VAV1  1.097638  0.272181
138        Resting          WT1  1.073121  0.312364
139        Resting      control  0.409309  0.096798

[140 rows x 4 columns]

In [4]:
distance_df.to_pickle('data/schmidt_distance_df.pkl')